# Lesson 01 - Увод у AI агенте

Добро дошли на прву лекцију у курсу **AI агенти за почетнике**!

**AI агент** је програм који користи велики језички модел (LLM) као свој мотор размишљања и може предузимати *акције* у стварном свету — позивати API-је, упитивати базе података или извршавати код — како би остварио циљ у име корисника.

У овом нотебуку ћете изградити свог првог агента: **Туристички агент** који препоручује дестинације за одмор. Усput ћете научити како да:

1. Повежете се са Azure AI Foundry Agent Service користећи **Microsoft Agent Framework**.
2. Дајете агенту **очек** — обичну Python функцију коју може позвати.
3. Покренете агента и испитате његов одговор.
4. Стримујете одговор агента токен по токен.


## Подешавање

Пре покретања ове свеске, уверите се да имате:

1. **Azure AI Foundry пројекат** са распоређеним моделом за ћаскање (нпр. `gpt-4o-mini`).
2. **Пријавили сте се помоћу Azure CLI** — покрените `az login` у вашем терминалу.
3. **Поставили потребне променљиве окружења:**
   - `AZURE_AI_PROJECT_ENDPOINT` — ваш крајњи пут Azure AI Foundry пројекта.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — име вашег распоређеног модела.

Доња ћелија инсталира Python пакете које су вам потребни.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Креирање Вашег Првог Агента

Агенту су потребне две ствари:

- **Упутства** која му говоре *ко је* и *како да се понаша* (системски подсетник).
- **Алати** — Python функције украшене са `@tool` које агент може позвати да би добио информације или извршио радње.

Испод дефинишемо једноставан алат који враћа листу популарних дестинација за годишњи одмор. Агент ће користити овај алат када корисник затражи препоруке за путовања.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Streaming Odgovori

Za interaktivnije iskustvo možete **streamovati** odgovor agenta. Umesto da čekate ceo odgovor, agent isporučuje delove teksta kako se generišu. Ovo je posebno korisno u chat interfejsima gde želite da prikažete izlaz u realnom vremenu.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Резиме

У овој лекцији сте научили како да:

- **Креирате провајдера** који се повезује на Azure AI Foundry Agent Service преко `AzureAIProjectAgentProvider`.
- **Дефинишете алат** користећи декоратер `@tool` тако да агент може позивати ваше Python функције.
- **Покренете агента** са корисничком поруком и испишете његов одговор.
- **Стримујете одговоре** за излаз у реалном времену.

У следећој лекцији ћемо детаљније истражити агентске оквире и научити како да агентима дамо моћније алате и могућности мулти-степ размишљања.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Одрицање од одговорности**:
Овај документ је преведен помоћу АИ услуге за превођење [Co-op Translator](https://github.com/Azure/co-op-translator). Иако се трудимо да обезбедимо тачност, молимо имајте у виду да аутоматизовани преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитетним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било какве неспоразуме или погрешне тумачења која произилазе из употребе овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
